In [19]:
%load_ext dotenv
%dotenv /home/aurora/.env
%matplotlib inline

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [20]:
import numpy as np
import pandas as pd
import datetime as dt
from google.cloud import bigquery
import re
import pymongo
import os
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from flatten_json import flatten
#warnings.filterwarnings('ignore')

In [21]:
# # Time
# start = dt.datetime(2019,4,1)
# end = dt.datetime(2019,5,5)
# print(start,end,end-start)

In [22]:
cursor = pymongo.MongoClient("mongodb://" + os.environ['user'] + ':' + 
                             os.environ['pass'] + '@' + os.environ['db1']  + "/?authSource=" + 
                             os.environ['dbname'])
c_users = cursor.superstars.users
a = c_users.find_one()
aw = []
for documents in c_users.find({},{'collectables'}):
    aw.append(documents)
dic_flattened = [flatten(d) for d in aw]
df = pd.DataFrame(dic_flattened)
users = df[["_id","collectables_soft_currency_0_quantity"]]
users.columns = ["user_id","mrp"]
print(len(users))
users.head()

128386


,user_id,mrp
0,5c702c13453bed5fa3bdba59,75
1,5c702f100899f358b8f31db7,85
2,5c702f1a0899f358b8f31dd6,60
3,5c702f9865a1f762ddc8cbe6,45
4,5c70308f0899f358b8f31e19,60


In [23]:
# mrp = []
# for user in aw:
#     for currency in user['collectables']['soft_currency']:
#         if(currency['id'] == 1):
#             mrp.append(currency['quantity'])     
# mrp

In [28]:
def function(row):
    if row['mrp'] < 20:
        return 0
    elif ((row['mrp'] >= 20) &(row['mrp'] < 50)):
        return 1
    elif ((row['mrp'] >= 50) &(row['mrp'] < 100)):
        return 2
    elif ((row['mrp'] >= 100) &(row['mrp'] < 200)):
        return 3
    elif ((row['mrp'] >= 200) &(row['mrp'] < 400)):
        return 4
    elif ((row['mrp'] >= 400) &(row['mrp'] < 900)):
        return 5
    elif ((row['mrp'] >= 900) &(row['mrp'] < 1700)):
        return 6
    elif ((row['mrp'] >= 1700) &(row['mrp'] < 3200)):
        return 7
    elif ((row['mrp'] >= 3200) &(row['mrp'] < 5700)):
        return 8
    elif ((row['mrp'] >= 5700) &(row['mrp'] < 10700)):
        return 9
    elif ((row['mrp'] >= 10700) &(row['mrp'] < 18700)):
        return 10
    elif ((row['mrp'] >= 18700) &(row['mrp'] < 30700)):
        return 11
    if ((row['mrp'] >= 30700) &(row['mrp'] < 48200)):
        return 12
    elif ((row['mrp'] >= 48200) &(row['mrp'] < 73200)):
        return 13
    elif ((row['mrp'] >= 73200) &(row['mrp'] < 100000)):
        return 14
    else:
        return 15

users['level'] = users.apply(function, axis=1)
users.head()

/home/aurora/miniconda3/lib/python3.7/site-packages/ipykernel_launcher.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy


,user_id,mrp,level
0,5c702c13453bed5fa3bdba59,75,2
1,5c702f100899f358b8f31db7,85,2
2,5c702f1a0899f358b8f31dd6,60,2
3,5c702f9865a1f762ddc8cbe6,45,1
4,5c70308f0899f358b8f31e19,60,2


In [30]:
f = {'user_id':'count','mrp':'sum'}
df = users.groupby("level").agg(f)
df['average points'] = (df['mrp']/df['user_id']).round()
df.columns = ['number of users','total mrp','average mrp rating']
df

,number of users,total mrp,average mrp rating
level,,,
0,455,565,1.0
1,85529,2368025,28.0
2,25871,1651490,64.0
3,7012,807235,115.0
4,3349,1022295,305.0
5,5386,2954920,549.0
6,662,756350,1143.0
7,118,238325,2020.0
8,2,8415,4208.0
